In [1]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import csv
import polars as pl
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo

BASE = Path(
    r"J:\Product & Client Solutions\Toolbox\Calculators"
    r"\Business Tools\Tender Summary\GAP Pricing Engine"
)
SOURCE       = BASE / "Basis and Assumptions.xlsx"
MASTER_XLSX  = BASE / "basis_master" / "basis_master_v2026_07.xlsx"
SNAPSHOT_DIR = BASE / "basis_snapshots" / "basis_v2026_07"
BUILD_LOG    = SNAPSHOT_DIR / "build_log.csv"

MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
assert SOURCE.exists()
print("✅ Setup ready")

✅ Setup ready


In [2]:
def find_anchors(ws, text: str) -> list[tuple[int, int]]:
    hits = []
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == text:
                hits.append((cell.row, cell.column))
    return hits


def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def as_str(v):
    """Coerce to trimmed string or None."""
    if v is None:
        return None
    s = str(v).strip()
    return s if s else None


def as_float(v):
    """Coerce to float or None."""
    if v is None:
        return None
    try:
        return float(v)
    except (ValueError, TypeError):
        return None


def as_int(v):
    """Coerce to int or None."""
    if v is None:
        return None
    try:
        return int(v)
    except (ValueError, TypeError):
        return None

print("✅ Helpers ready")

✅ Helpers ready


In [3]:
print(f"📂 Opening source: {SOURCE.name}\n")
wb = openpyxl.load_workbook(SOURCE, data_only=True)
all_rows = []

# ============================================================
# 1. GLA industry mapping + factors
# ============================================================
print("🔎 GLA industry (mapping + factors, per short_code, 2 scheme sizes)...")
ws = wb["GLA Basis"]
r0, c0 = find_anchors(ws, "Short Code")[0]
r = r0 + 1
gla_count = 0
while r < 260:
    sc = ws.cell(row=r, column=c0).value
    if sc is None:
        break
    prim = as_str(ws.cell(row=r, column=c0+1).value)
    sec  = as_str(ws.cell(row=r, column=c0+2).value)
    grp  = as_str(ws.cell(row=r, column=c0+3).value)
    gn   = as_int(ws.cell(row=r, column=c0+4).value)
    for scheme_size, inc_col, acaf_col in [("small", c0+5, c0+7),
                                            ("large", c0+6, c0+8)]:
        all_rows.append(dict(
            product="gla",
            short_code=as_str(sc),
            primary=prim, secondary=sec, grouping=grp, group_number=gn,
            scheme_size=scheme_size,
            inc_adj=as_float(ws.cell(row=r, column=inc_col).value),
            acaf_adj=as_float(ws.cell(row=r, column=acaf_col).value),
            cls=None,
        ))
    gla_count += 1
    r += 1
print(f"  ✅ gla   → {gla_count} short codes × 2 scheme_sizes = {gla_count*2} rows")

# ============================================================
# 2. LSDB industry mapping + factors
# ============================================================
print("\n🔎 LSDB industry (mapping + factors)...")
ws = wb["LSDB Basis"]
r0, c0 = find_anchors(ws, "Primary")[0]
r = r0 + 1
lsdb_count = 0
while r < 260:
    p = ws.cell(row=r, column=c0).value
    if p is None:
        break
    all_rows.append(dict(
        product="lsdb",
        short_code=None,
        primary=as_str(p),
        secondary=as_str(ws.cell(row=r, column=c0+1).value),
        grouping=as_str(ws.cell(row=r, column=c0+2).value),
        group_number=None,
        scheme_size=None,
        inc_adj=as_float(ws.cell(row=r, column=c0+3).value),
        acaf_adj=as_float(ws.cell(row=r, column=c0+4).value),
        cls=None,
    ))
    lsdb_count += 1
    r += 1
print(f"  ✅ lsdb  → {lsdb_count} primary/secondary combinations")

# ============================================================
# 3. CI industry → class mapping
# ============================================================
print("\n🔎 CI industry → class mapping...")
ws = wb["CI Basis"]
r0, c0 = find_anchors(ws, "Industry List")[0]
r = r0 + 1
ci_map_count = 0
while r < 250:
    g = ws.cell(row=r, column=c0).value
    cls = ws.cell(row=r, column=c0+1).value
    if g is None or cls is None:
        break
    all_rows.append(dict(
        product="ci",
        short_code=None, primary=None, secondary=None,
        grouping=as_str(g), group_number=None, scheme_size=None,
        inc_adj=None, acaf_adj=None, cls=as_str(cls),
    ))
    ci_map_count += 1
    r += 1
print(f"  ✅ ci    → {ci_map_count} groupings mapped to classes")

# ============================================================
# 4. CI class factors — Old and New lists
# ============================================================
print("\n🔎 CI class A-E factors (Old + New)...")
industry_anchors = [(r, c) for (r, c) in find_anchors(ws, "Industry") if r == 153]
industry_anchors.sort(key=lambda rc: rc[1])
for (r0, c0), pkey in zip(industry_anchors, ["ci_old", "ci_new"]):
    r = r0 + 1
    n = 0
    for _ in range(5):
        cls = ws.cell(row=r, column=c0).value
        fac = ws.cell(row=r, column=c0+1).value
        if cls is None or fac is None:
            break
        all_rows.append(dict(
            product=pkey,
            short_code=None, primary=None, secondary=None,
            grouping=as_str(cls), group_number=None, scheme_size=None,
            inc_adj=as_float(fac), acaf_adj=None, cls=None,
        ))
        n += 1
        r += 1
    print(f"  ✅ {pkey} → {n} classes")

# ============================================================
# 5. CI hazard occupation loadings (→ ci_acc + ci_std)
# ============================================================
print("\n🔎 CI hazard occupations (→ ci_acc + ci_std)...")
r0, c0 = find_anchors(ws, "Industry factors")[0]
r = r0 + 1
hazard = []
for _ in range(5):
    g = ws.cell(row=r, column=c0).value
    fac = ws.cell(row=r, column=c0+1).value
    if g is None or fac is None:
        break
    hazard.append((as_str(g), as_float(fac)))
    r += 1
for pkey in ["ci_acc", "ci_std"]:
    for g, fac in hazard:
        all_rows.append(dict(
            product=pkey,
            short_code=None, primary=None, secondary=None,
            grouping=g, group_number=None, scheme_size=None,
            inc_adj=fac, acaf_adj=None, cls=None,
        ))
print(f"  ✅ ci_acc + ci_std → {len(hazard)} groupings each = {len(hazard)*2} rows")

# ============================================================
# 6. Spouse hazard occupation loadings
# ============================================================
print("\n🔎 Spouse hazard occupations (→ spouse)...")
ws = wb["Spouse Basis"]
r0, c0 = find_anchors(ws, "Industry factors")[0]
r = r0 + 1
sp_count = 0
for _ in range(5):
    g = ws.cell(row=r, column=c0).value
    fac = ws.cell(row=r, column=c0+1).value
    if g is None or fac is None:
        break
    all_rows.append(dict(
        product="spouse",
        short_code=None, primary=None, secondary=None,
        grouping=as_str(g), group_number=None, scheme_size=None,
        inc_adj=as_float(fac), acaf_adj=None, cls=None,
    ))
    sp_count += 1
    r += 1
print(f"  ✅ spouse → {sp_count} groupings")

# ============================================================
# Assemble
# ============================================================
df = pl.DataFrame(all_rows, infer_schema_length=None) \
       .rename({"cls": "class"}) \
       .sort(["product", "grouping", "short_code"])

📂 Opening source: Basis and Assumptions.xlsx

🔎 GLA industry (mapping + factors, per short_code, 2 scheme sizes)...
  ✅ gla   → 108 short codes × 2 scheme_sizes = 216 rows

🔎 LSDB industry (mapping + factors)...
  ✅ lsdb  → 106 primary/secondary combinations

🔎 CI industry → class mapping...
  ✅ ci    → 27 groupings mapped to classes

🔎 CI class A-E factors (Old + New)...
  ✅ ci_old → 5 classes
  ✅ ci_new → 5 classes

🔎 CI hazard occupations (→ ci_acc + ci_std)...
  ✅ ci_acc + ci_std → 3 groupings each = 6 rows

🔎 Spouse hazard occupations (→ spouse)...
  ✅ spouse → 3 groupings


In [4]:
SANITY_CHECKS = [
    # (product, short_code, scheme_size, grouping, expected_inc_adj, expected_acaf_or_None, expected_class_or_None)
    ("gla",     "AGFHR", "small", None,                                                    1.00, 1.00, None),
    ("gla",     "AGFHR", "large", None,                                                    1.00, 1.00, None),
    ("gla",     "MINCP", "small", None,                                                    1.70, 1.00, None),
    ("gla",     "MINCP", "large", None,                                                    1.05, 1.15, None),
    ("gla",     "PADMN", "small", None,                                                    1.15, 1.10, None),
    ("gla",     "PADMN", "large", None,                                                    1.50, 1.00, None),
    ("lsdb",    None,    None,    "Agriculture, Forestry and Fishing",                     0.95, 0.90, None),
    ("lsdb",    None,    None,    "Mining, Resource Extraction & Quarrying",               1.45, 1.00, None),
    ("ci_old",  None,    None,    "A",                                                     0.6666666666666669, None, None),
    ("ci_old",  None,    None,    "E",                                                     0.96, None, None),
    ("ci_new",  None,    None,    "A",                                                     0.8333333333333335, None, None),
    ("ci_new",  None,    None,    "E",                                                     1.20, None, None),
    ("ci_acc",  None,    None,    "Underground mining",                                    2.23, None, None),
    ("ci_std",  None,    None,    "Longdistance transport",                                2.23, None, None),
    ("spouse",  None,    None,    "Other",                                                 1.00, None, None),
    ("ci",      None,    None,    "Agriculture, Forestry and Fishing",                     None, None, "D"),
    ("ci",      None,    None,    "Cleaning & Maintenance Services",                       None, None, "C"),
]

print("🧪 Sanity checks:\n")
all_pass = True
for product, sc, scheme, grp, exp_inc, exp_acaf, exp_cls in SANITY_CHECKS:
    q = pl.col("product") == product
    if sc is not None:
        q = q & (pl.col("short_code") == sc)
    if scheme is not None:
        q = q & (pl.col("scheme_size") == scheme)
    if grp is not None:
        q = q & (pl.col("grouping") == grp)
    hit = df.filter(q)
    if hit.is_empty():
        print(f"  ❌ NOT FOUND  {product:<8} sc={sc} scheme={scheme} grp='{str(grp)[:40]}'")
        all_pass = False
        continue
    row = hit.row(0, named=True)
    inc_ok  = (exp_inc  is None) or (row["inc_adj"]  is not None and abs(row["inc_adj"]  - exp_inc)  < 1e-6)
    acaf_ok = (exp_acaf is None) or (row["acaf_adj"] is not None and abs(row["acaf_adj"] - exp_acaf) < 1e-6)
    cls_ok  = (exp_cls  is None) or (row["class"]    == exp_cls)
    ok = inc_ok and acaf_ok and cls_ok
    status = "✅ MATCH" if ok else "❌ MISMATCH"
    if not ok:
        all_pass = False
    print(f"  {status}  {product:<8} sc={str(sc):<7} scheme={str(scheme):<5} "
          f"grp='{str(row['grouping'])[:35]:<35}' inc={row['inc_adj']} acaf={row['acaf_adj']} cls={row['class']}")

print(f"\n{'🎉 ALL PASSED' if all_pass else '⚠️ FIX FAILURES BEFORE WRITING'}")

🧪 Sanity checks:

  ✅ MATCH  gla      sc=AGFHR   scheme=small grp='Agriculture - Farming, Hunting and ' inc=1.0 acaf=1.0 cls=None
  ✅ MATCH  gla      sc=AGFHR   scheme=large grp='Agriculture - Farming, Hunting and ' inc=1.0 acaf=1.0 cls=None
  ✅ MATCH  gla      sc=MINCP   scheme=small grp='Mining, Resource Extraction & Quarr' inc=1.7 acaf=1.0 cls=None
  ✅ MATCH  gla      sc=MINCP   scheme=large grp='Mining, Resource Extraction & Quarr' inc=1.05 acaf=1.15 cls=None
  ✅ MATCH  gla      sc=PADMN   scheme=small grp='Municipalities                     ' inc=1.15 acaf=1.1 cls=None
  ✅ MATCH  gla      sc=PADMN   scheme=large grp='Municipalities                     ' inc=1.5 acaf=1.0 cls=None
  ✅ MATCH  lsdb     sc=None    scheme=None  grp='Agriculture, Forestry and Fishing  ' inc=0.95 acaf=0.9 cls=None
  ✅ MATCH  lsdb     sc=None    scheme=None  grp='Mining, Resource Extraction & Quarr' inc=1.45 acaf=1.0 cls=None
  ✅ MATCH  ci_old   sc=None    scheme=None  grp='A                               

In [5]:
# ---------- Guard: Excel file lock ----------
if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(
            f"❌ '{MASTER_XLSX.name}' is open in Excel. Close it and re-run.")

# ---------- 0. Teardown: remove old industry_factors artifacts ----------
print("🗑️  Removing old 'industry_factors' artifacts...")

# Delete parquet
old_parquet = SNAPSHOT_DIR / "industry_factors.parquet"
if old_parquet.exists():
    old_parquet.unlink()
    print(f"  ✅ Deleted {old_parquet.name}")

# Delete sheet from master xlsx
if MASTER_XLSX.exists():
    wb_out = openpyxl.load_workbook(MASTER_XLSX)
    if "industry_factors" in wb_out.sheetnames:
        del wb_out["industry_factors"]
        wb_out.save(MASTER_XLSX)
        print(f"  ✅ Deleted sheet 'industry_factors' from master xlsx")

# ---------- 1. Parquet snapshot ----------
parquet_path = SNAPSHOT_DIR / "industry.parquet"
df.write_parquet(parquet_path)
parquet_size = parquet_path.stat().st_size / 1024
parquet_hash = sha256_of(parquet_path)
print(f"\n📦 Parquet: {parquet_path.name}  ({parquet_size:.1f} KB)")

# ---------- 2. Master xlsx sheet ----------
if MASTER_XLSX.exists():
    wb_out = openpyxl.load_workbook(MASTER_XLSX)
    if "industry" in wb_out.sheetnames:
        del wb_out["industry"]
else:
    wb_out = openpyxl.Workbook()
    if "Sheet" in wb_out.sheetnames:
        del wb_out["Sheet"]

ws = wb_out.create_sheet("industry")
headers = ["product", "short_code", "primary", "secondary", "grouping",
           "group_number", "scheme_size", "inc_adj", "acaf_adj", "class"]
ws.append(headers)

# Style the header row (row 1) — iterate ws to hit each header cell
header_row = next(ws.iter_rows(min_row=1, max_row=1))
for cell in header_row:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="305496")
    cell.alignment = Alignment(horizontal="left")

for row in df.iter_rows():
    ws.append(row)

widths = {"A": 10, "B": 12, "C": 40, "D": 55, "E": 45,
          "F": 8, "G": 12, "H": 10, "I": 10, "J": 8}
for col, w in widths.items():
    ws.column_dimensions[col].width = w
ws.freeze_panes = "A2"

table = Table(displayName="tbl_industry", ref=f"A1:J{ws.max_row}")
table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
ws.add_table(table)
wb_out.save(MASTER_XLSX)
print(f"📄 Master xlsx: sheet 'industry' → {df.height} rows")

# ---------- 3. Append build log ----------
source_hash = sha256_of(SOURCE)
timestamp   = datetime.now(timezone.utc).isoformat(timespec="seconds")

teardown_log = {
    "timestamp_utc":     timestamp,
    "sheet":             "industry_factors",
    "row_count":         0,
    "distinct_products": 0,
    "source_file":       SOURCE.name,
    "source_sha256":     source_hash,
    "parquet_file":      "",
    "parquet_sha256":    "",
    "parquet_kb":        0,
    "status":            "removed (merged into 'industry')",
    "built_by":          "Mohamed Hussein Gadiwala",
}

log_row = {
    "timestamp_utc":     timestamp,
    "sheet":             "industry",
    "row_count":         df.height,
    "distinct_products": df["product"].n_unique(),
    "source_file":       SOURCE.name,
    "source_sha256":     source_hash,
    "parquet_file":      parquet_path.name,
    "parquet_sha256":    parquet_hash,
    "parquet_kb":        round(parquet_size, 2),
    "status":            "ok",
    "built_by":          "Mohamed Hussein Gadiwala",
}

log_exists = BUILD_LOG.exists()
with open(BUILD_LOG, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(log_row.keys()))
    if not log_exists:
        writer.writeheader()
    writer.writerow(teardown_log)
    writer.writerow(log_row)

print(f"📝 Build log:  appended 2 rows (teardown + new build)")
print(f"\n🎉 Done!")

🗑️  Removing old 'industry_factors' artifacts...

📦 Parquet: industry.parquet  (8.4 KB)
📄 Master xlsx: sheet 'industry' → 368 rows
📝 Build log:  appended 2 rows (teardown + new build)

🎉 Done!
